# OpenPlaque — confirmed LAD segment proximal backtrack

This experiment starts from the convincing portion of the prior LAD result instead of searching for the LAD or left-main origin from scratch. The prior LAD segment at approximately **32.5–55.5 mm** is first rechecked with the previously validated RCA calibration. If it passes, that segment is frozen and a short source-resolution beam search tracks only **proximally** toward the LAD takeoff.

The TotalSegmentator **aorta mask is used only as an anatomical constraint**. It is not a coronary segmentation. No LCX tracking and no plaque processing are performed.


## Step 1 — Mount Google Drive


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache reuse controls


In [ ]:
REUSE_SOURCE_CT = True
REUSE_RCA_CALIBRATION = True
REUSE_FROZEN_LAD = True
REUSE_AORTA_CONSTRAINT = True
REUSE_BACKTRACK = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install this branch


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch lad-confirmed-segment-backtrack-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy matplotlib pandas psutil "pylibjpeg>=2.0" "pylibjpeg-libjpeg>=2.1"
import sys, os, gc, psutil
sys.path.insert(0, '/content/OpenPlaque/src')
def ram(label):
    p = psutil.Process(os.getpid())
    print(f'{label}: RSS {p.memory_info().rss/1024**3:.2f} GB')
ram('After install')


## Step 4 — Initialize workflow


In [ ]:
from openplaque.lad_confirmed_backtrack import LADConfirmedBacktrackWorkflow, ALGORITHM_VERSION
reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'rca_calibration': REUSE_RCA_CALIBRATION,
    'frozen_lad': REUSE_FROZEN_LAD,
    'aorta_constraint': REUSE_AORTA_CONSTRAINT,
    'backtrack': REUSE_BACKTRACK,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = LADConfirmedBacktrackWorkflow('/content/drive/MyDrive/OpenPlaque', reuse=reuse)
print('Algorithm:', ALGORITHM_VERSION)
display(wf.cache_status())


## Step 5 — Load disk-backed source CCTA

The workflow first looks for the source-CCTA memmap created by earlier experiments. If none is available it streams series 7 slice-by-slice.


In [ ]:
wf.load_source_ct()
gc.collect(); ram('After source CT')
print('Source CT shape:', wf.ct.shape)
print('Spacing z,y,x (mm):', wf.spacing)


## Step 6 — Load the previously validated RCA calibration

This step deliberately does **not** recalculate the RCA calibration that was corrupted in the previous LAD run. It imports the strong staged-left-main calibration when available and checks it against a plausibility envelope; otherwise it uses the pinned validated values from that run.


In [ ]:
rca = wf.load_validated_rca_calibration()
display(rca)


## Step 7 — Revalidate and freeze the convincing LAD segment

The 32.5–55.5 mm portion of the previous best LAD path is remeasured in true source-volume orthogonal planes using the corrected RCA reference. Backtracking is not allowed unless this segment passes.


In [ ]:
frozen = wf.freeze_confirmed_lad(start_arc_mm=32.5, end_arc_mm=55.5)
print('Frozen LAD summary:')
display(frozen)
display(wf.frozen_qc)
gc.collect(); ram('After frozen LAD validation')


## Step 8 — Build the TotalSegmentator aorta constraint

Only a small crop of the existing TotalSegmentator aorta mask is retained. It prevents the proximal search from entering the aorta and provides distance-to-aorta information.


In [ ]:
wf.build_aorta_constraint()
gc.collect(); ram('After aorta constraint')
print('Aorta-constraint crop shape:', wf.aorta_crop.shape)


## Step 9 — Backtrack proximally from the frozen LAD

A short source-resolution beam search advances only from the proximal end of the validated LAD segment. Every step is scored from its **actual orthogonal lumen plane**, with a continuity constraint and a weak bias toward the aortic-root reference. Broad chambers and off-center structures should therefore terminate the search rather than attract it.


In [ ]:
summary = wf.backtrack(step_mm=0.80, max_backtrack_mm=24.0, beam_width=5)
print('Backtrack summary:')
display(summary)
print('Estimated LAD origin/takeoff neighborhood:')
display(wf.origin)
display(wf.backtrack_qc)
gc.collect(); ram('After proximal backtrack')


## Step 10 — QC figures

The decisive figures are: corrected frozen-LAD cross-sections, proximal backtrack cross-sections, combined source MIPs, the radius/offset/score profiles, and the first 10 mm around the estimated proximal LAD takeoff.


In [ ]:
figs = wf.plot_qc()
for f in figs:
    print('Saved:', f)
gc.collect(); ram('After QC figures')


## Step 11 — Package report-back ZIP


In [ ]:
zip_path = wf.package()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LAD_CONFIRMED_BACKTRACK_REPORT_BACK.zip')


## Step 12 — Report back

After the ZIP is written, return to ChatGPT and say **Retrieve and analyze**. The estimated proximal endpoint is not accepted as the LAD takeoff until the source-resolution QC images are visually convincing.
